In [1]:
# NLP303 Assessment 3
#
# Detecting AI-generated Text
# Using Transformer-Based Classification
#
#
#
# Tibor Titusz Tarcsai - A00121308
#
# Jonathan Lim - A00142089
#
# Thomas Galindo Salazar - A00129258
#
#
#
# The purpose of this implementation is to demonstrate the building of a working protoype for a Classification task based on the Assessment 2 proposal.
#
#
#***********************
# HOW TO RUN THE CODE:
#***********************
#
# 1. - The notebook can run either in Jupyter Notebook or Google Colab 
#    - If using Google Colab, a Google Drive account is required for data access and storage.
#      Link for running on Colab: 
#
# 2. All helper functions are imported and loaded from the src folder
#
# 3. In Section [2], the "find_cat_dog_lion_tiger_folders" path finding function automatically locates the dataset folder,
#    so NO manual path configuration is required from the user.

# Environment Setup and Dependencies

In [1]:
!pip install -r ../requirements.txt
print("### Dependencies installed successfully ###")

### Dependencies installed successfully ###


In [2]:
# Initial Library Imports
# --------------------------------
# - torch for ... implementation
#
#
#
#
#

import os
import torch
import pandas as pd
import transformers

from torchinfo import summary
from transformers import pipeline

import sys
sys.path.append("../")


from sklearn.model_selection import train_test_split

from transformers import AutoTokenizer

from src.downloader import DatasetDownloader

from src.datasetbuilder import DatasetBuilder

from src.preprocessing import TextPreprocessor

from src.dataset_wrapper import DatasetWrapper

from transformers import AutoTokenizer, AutoModelForSequenceClassification

I0000 00:00:1785923470.535064   29574 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


# Section 1 - Load Dataset

In [4]:
# Initialises the downloader class
downloader = DatasetDownloader()

# Downloads raw datasets from google drive
print("### Downloading Dataset from Google Drive... ###")
downloader.start_downloading_dataset()

### Downloading Dataset from Google Drive... ###


Downloading...
From: https://drive.google.com/uc?id=1v8ZKV3p6KLDMsOscVLj1Z5zgNYGJfaQp
To: /home/titus/projects/roberta-ai-text-detector/data/humanised_v2_first_2400.csv
100%|██████████| 7.48M/7.48M [00:01<00:00, 5.51MB/s]
Downloading...
From: https://drive.google.com/uc?id=1U9Lhpo2qet7dPAuswxHFGAJEggc26bR0
To: /home/titus/projects/roberta-ai-text-detector/data/ai_polished_v2_first_2400.csv
100%|██████████| 7.36M/7.36M [00:01<00:00, 5.51MB/s]
Downloading...
From: https://drive.google.com/uc?id=182-e58HGw67tacTudS7wacm6DZZCbMEZ
To: /home/titus/projects/roberta-ai-text-detector/data/pure_ai_v2_first_2400.csv
100%|██████████| 4.85M/4.85M [00:00<00:00, 5.81MB/s]
Downloading...
From: https://drive.google.com/uc?id=15BDFQcaylNmK6Uy-BaKKe__jXQ6MgdmK
To: /home/titus/projects/roberta-ai-text-detector/data/pure_human_v2_first_2400.csv
100%|██████████| 3.96M/3.96M [00:00<00:00, 5.90MB/s]


In [4]:
# Restructures datasets


raw_dataset_paths = [
            {
                "pure_human": "../data/pure_human_v2_first_2400.csv"
            },
            {
                "pure_ai": "../data/pure_ai_v2_first_2400.csv"
            },
            {
                "ai_polished": "../data/ai_polished_v2_first_2400.csv"
            },
            {
                "humanised": "../data/humanised_v2_first_2400.csv"
            },
        ]


builder = DatasetBuilder()

raw_dataset = builder.build_dataset(
    raw_dataset_paths[0]["pure_human"],     # Label 0
    raw_dataset_paths[1]["pure_ai"],        # Label 1
    raw_dataset_paths[2]["ai_polished"],    # Label 2
    raw_dataset_paths[3]["humanised"],      # Label 3
)


In [5]:
print(len(raw_dataset))

9600


In [6]:
raw_dataset.iloc[7202].cleaned_text

AttributeError: 'Series' object has no attribute 'cleaned_text'

In [7]:
# Selects rows 
raw_dataset.iloc[7200:7250]


,text,label,label_name
7200,"A recent paper titled ""Future-AI: Guiding Prin...",3,ai_written_humanised
7201,The phenomenon of matrix singularity has garne...,3,ai_written_humanised
7202,Protecting secret messages sent over anonymous...,3,ai_written_humanised
7203,The phenomenon of sputtering on polar surfaces...,3,ai_written_humanised
7204,\nThe introduction of two new scheduling primi...,3,ai_written_humanised
7205,"""A quantum mechanical description of anisotrop...",3,ai_written_humanised
7206,The development of a self-perfect absorber has...,3,ai_written_humanised
7207,Lagrangian manifolds in Hilbert space have bee...,3,ai_written_humanised
7208,Researchers investigated the upper limits on g...,3,ai_written_humanised
7209,The article examines the supersymmetric topolo...,3,ai_written_humanised


# Section 2 - Text Preprocessing

In [8]:
# Drops rows where 'text' is missing
raw_dataset = raw_dataset.dropna(subset=["text"]).reset_index(drop=True)

print(len(raw_dataset))

9600


In [9]:
# Initialises the TextPreprocessor class for text cleaning
text_preprocessor = TextPreprocessor()

raw_dataset["cleaned_text"] = raw_dataset["text"].apply(text_preprocessor.clean_text)



print(raw_dataset[["cleaned_text", "text"]].iloc[7204])

cleaned_text    The introduction of two new scheduling primiti...
text            \nThe introduction of two new scheduling primi...
Name: 7204, dtype: str


# Section 3 - Data Split

In [10]:
# Shuffles the dataset and stores features and labels
dataset_shuffled = raw_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

features = dataset_shuffled['cleaned_text']
labels = dataset_shuffled['label']


# Splits the data into an initial 80-20 split
x_train_raw, x_temp_raw, y_train, y_temp = train_test_split(
    features,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

# Split the remaining data into validation and test (10-10)
x_val_raw, x_test_raw, y_val, y_test = train_test_split(
    x_temp_raw,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [11]:
# Section 3 - Tokenisation (BPE)

"""""
In this section during tokenisation the encodings are created to suit the basemodel with 512 token limit.
"""""

# Imports same tokeniser
tokenizer = AutoTokenizer.from_pretrained("fakespot-ai/roberta-base-ai-text-detection-v1")

# Creates input encodings
train_encodings = tokenizer(x_train_raw.to_list(), truncation=True, padding="max_length", max_length=512)
val_encodings = tokenizer(x_val_raw.to_list(), truncation=True, padding="max_length", max_length=512)
test_encodings = tokenizer(x_test_raw.to_list(), truncation=True, padding="max_length", max_length=512)

# Transforms encodings into Pytorch tensors
X_train = DatasetWrapper(train_encodings, y_train)
X_val = DatasetWrapper(val_encodings, y_val)
X_test = DatasetWrapper(test_encodings, y_temp)


In [18]:
# Section 4- Fine-tuning - Part 1
# REFERENCE: https://huggingface.co/transformers/v3.2.0/custom_datasets.html

from torch.utils.data import DataLoader
from torch.optim import AdamW 

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')


model = AutoModelForSequenceClassification.from_pretrained(
    "fakespot-ai/roberta-base-ai-text-detection-v1",
    num_labels=4,
    ignore_mismatched_sizes=True)

# Prints model summary
summary(model, input_size=(1, 512), dtypes=[torch.long])


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at fakespot-ai/roberta-base-ai-text-detection-v1 and are newly initialized because the shapes did not match:
- classifier.out_proj.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.out_proj.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Layer (type:depth-idx)                                            Output Shape              Param #
RobertaForSequenceClassification                                  [1, 4]                    --
├─RobertaModel: 1-1                                               [1, 512, 768]             --
│    └─RobertaEmbeddings: 2-1                                     [1, 512, 768]             --
│    │    └─Embedding: 3-1                                        [1, 512, 768]             38,603,520
│    │    └─Embedding: 3-2                                        [1, 512, 768]             768
│    │    └─Embedding: 3-3                                        [1, 512, 768]             394,752
│    │    └─LayerNorm: 3-4                                        [1, 512, 768]             1,536
│    │    └─Dropout: 3-5                                          [1, 512, 768]             --
│    └─RobertaEncoder: 2-2                                        [1, 512, 768]             --
│    │    └─ModuleList: 3-6 

In [19]:
# Fine-tuning - Part 2

model.to(device)
model.train()

train_loader = DataLoader(X_train, batch_size=16, shuffle=True)

optim = AdamW(model.parameters(), lr=5e-5)

print("Fine tuning started...")

for epoch in range(3):

    total_loss = 0.0
    for batch_idx, batch in enumerate(train_loader):
        optim.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        
        loss = outputs.loss

        loss.backward()
        optim.step()

        total_loss += loss.item()

        if batch_idx % 50 ==0:
            print(f"Epoch: {epoch+1} | batch: {batch_idx}/{len(train_loader)} | Running batch loss: {loss.item():.4f}")

model.eval()
print("Fine tuning is completed.")

Fine tuning started...
Epoch: 1 | batch: 0/480 | Running batch loss: 1.5134


KeyboardInterrupt: 

In [ ]:
# Saves the model weights and the tokenizer configurations locally
# *** Uncomment the below ONLY if model has to be saved ***.

# model.save_pretrained("../model/v1/")
# tokenizer.save_pretrained("../model/v1/")


💾 Success! Your optimized 4-class detector is safely backed up to ../model/v1/


In [ ]:
# Section 5 - Model Inference

In [ ]:
# Section 7 - Threshold Classification

In [ ]:
# Section 8 - Evaluation

In [ ]:
# Section 9 - Visualisations

Layer (type:depth-idx)                                       Output Shape              Param #
RobertaForSequenceClassification                             [1, 2]                    --
├─RobertaModel: 1-1                                          [1, 512, 768]             --
│    └─RobertaEmbeddings: 2-1                                [1, 512, 768]             --
│    │    └─Embedding: 3-1                                   [1, 512, 768]             38,603,520
│    │    └─Embedding: 3-2                                   [1, 512, 768]             768
│    │    └─Embedding: 3-3                                   [1, 512, 768]             394,752
│    │    └─LayerNorm: 3-4                                   [1, 512, 768]             1,536
│    │    └─Dropout: 3-5                                     [1, 512, 768]             --
│    └─RobertaEncoder: 2-2                                   [1, 512, 768]             --
│    │    └─ModuleList: 3-6                                  --               

In [ ]:
df = pd.read_csv("data/HUMAN_written_then_AI_polished.csv")

In [ ]:
print(df.head(200))

                                      id  \
0   e5e058ce-be2b-459d-af36-32532aaba5ff   
1   f95b107b-d176-4af5-90f7-4d0bb20caf93   
2   856d8972-9e3d-4544-babc-0fe16f21e04d   
3   fbc8a5ea-90fa-47b8-8fa7-73dd954f1524   
4   72c41b8d-0069-4886-b734-a4000ffca286   
5   72fe360b-cce6-4daf-b66a-1d778f5964f8   
6   df594cf4-9a0c-4488-bcb3-68f41e2d5a16   
7   853c0e51-7dd5-4bb5-8286-e4aa8820173b   
8   1649f195-8f98-4c79-92b6-54a5ca9261fa   
9   5e23ab14-b85f-48e8-9aa3-15452e73524e   
10  ddcb207c-a790-4e16-a053-4aced58d7c15   
11  b00bf7dc-4de9-4ab4-9962-a16e0b5f4628   
12  04d3809c-0abe-4bee-b1d2-9787af95362f   
13  06bffeb2-bea0-4b0b-b60d-767ba9b660a7   
14  5eb88a59-eb5a-49ea-8304-f67efe338921   
15  1389aa64-25fb-4e56-9358-ef34143bfea9   
16  d0064195-c22e-4550-a265-6b372deea3e0   
17  417afaa2-2d21-4df1-953b-768647de9980   
18  ce898c28-428f-446f-975e-a1265942f2da   
19  380cd71d-3300-422c-9cde-8a63e71f2797   
20  c093400c-2bd2-4e0d-a732-f99d499d58a9   
21  05f40b6d-67cf-4a6e-ad2f-cfe0